# LangChain 记忆系统 (Memory)

> 记忆系统让 LLM 拥有对话上下文能力

### 记忆类型：
1. **ConversationBufferMemory** - 缓冲记忆（存储全部消息）
2. **ConversationBufferWindowMemory** - 滑动窗口记忆（存储最近K条）
3. **ConversationSummaryMemory** - 摘要记忆（压缩对话历史）
4. **ConversationSummaryBufferMemory** - 摘要+缓冲混合记忆
5. **ConversationEntityMemory** - 实体记忆（提取关键实体）
6. **VectorStoreRetrieverMemory** - 向量存储记忆（语义检索）

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationSummaryBufferMemory,
    ConversationEntityMemory,
)
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 创建 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0
)

print("LLM 初始化完成")

## 1. ConversationBufferMemory - 缓冲记忆

存储所有对话历史，简单但会占用越来越多的 token

In [ ]:
# 创建缓冲记忆
buffer_memory = ConversationBufferMemory(return_messages=True)

# 创建对话链
conversation = ConversationChain(
    llm=llm,
    memory=buffer_memory,
    verbose=True
)

rprint("[bold]ConversationBufferMemory 测试[/bold]\n")

# 第一轮对话
response = conversation.predict(input="你好，我叫小明")
rprint(f"AI: {response}\n")

# 第二轮对话
response = conversation.predict(input="我是一名Python开发者")
rprint(f"AI: {response}\n")

# 第三轮对话（测试记忆）
response = conversation.predict(input="你还记得我的名字吗？")
rprint(f"AI: {response}\n")

# 查看记忆内容
rprint("[cyan]记忆内容:[/cyan]")
rprint(buffer_memory.load_memory_variables({}))

## 2. ConversationBufferWindowMemory - 滑动窗口记忆

只保留最近 K 轮对话，控制 token 使用

In [ ]:
# 创建滑动窗口记忆（只保留最近2轮）
window_memory = ConversationBufferWindowMemory(k=2, return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

rprint("[bold]ConversationBufferWindowMemory 测试 (k=2)[/bold]\n")

# 进行多轮对话
topics = [
    "我叫小明",
    "我在北京工作",
    "我喜欢Python编程",
    "我养了一只猫叫咪咪",
]

for topic in topics:
    response = conversation.predict(input=topic)
    rprint(f"输入: {topic}")
    rprint(f"AI: {response}\n")

# 测试记忆（最早的信息应该被遗忘）
response = conversation.predict(input="你还记得我的名字吗？")
rprint(f"测试记忆: 你还记得我的名字吗？")
rprint(f"AI: {response}")
rprint("[dim](由于窗口大小为2，最早的名字信息可能已被遗忘)[/dim]\n")

# 查看当前记忆
rprint("[cyan]当前记忆窗口:[/cyan]")
rprint(window_memory.load_memory_variables({}))

## 3. ConversationSummaryMemory - 摘要记忆

将对话历史压缩成摘要，节省 token

In [ ]:
# 创建摘要记忆
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=False
)

rprint("[bold]ConversationSummaryMemory 测试[/bold]\n")

# 进行多轮对话
conversations = [
    "你好，我叫小明，是一名软件工程师",
    "我在一家AI公司工作，主要负责后端开发",
    "我最近在学习LangChain框架",
    "我的爱好是打篮球和看电影",
]

for msg in conversations:
    response = conversation.predict(input=msg)
    rprint(f"输入: {msg}")
    rprint(f"AI: {response}\n")

# 查看摘要
rprint("[cyan]对话摘要:[/cyan]")
rprint(summary_memory.load_memory_variables({}))

## 4. ConversationSummaryBufferMemory - 摘要+缓冲混合记忆

结合摘要和缓冲，最近的消息保留原文，旧消息压缩成摘要

In [ ]:
# 创建混合记忆
hybrid_memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=200,  # 超过200 token时开始摘要
    return_messages=True
)

conversation = ConversationChain(
    llm=llm,
    memory=hybrid_memory,
    verbose=False
)

rprint("[bold]ConversationSummaryBufferMemory 测试[/bold]\n")

# 进行多轮对话
messages = [
    "我是一名数据科学家",
    "我使用Python和R进行数据分析",
    "我最近在研究自然语言处理",
    "我正在做一个情感分析项目",
    "这个项目使用了BERT模型",
]

for msg in messages:
    response = conversation.predict(input=msg)
    rprint(f"输入: {msg}")
    rprint(f"AI: {response[:80]}...\n")

# 查看混合记忆
rprint("[cyan]混合记忆内容:[/cyan]")
memory_vars = hybrid_memory.load_memory_variables({})
rprint(memory_vars)

## 5. ConversationEntityMemory - 实体记忆

提取并记住对话中的关键实体（人名、地点、组织等）

In [ ]:
# 创建实体记忆
entity_memory = ConversationEntityMemory(
    llm=llm,
    return_messages=True
)

conversation = ConversationChain(
    llm=llm,
    memory=entity_memory,
    verbose=False
)

rprint("[bold]ConversationEntityMemory 测试[/bold]\n")

# 包含多个实体的对话
messages = [
    "小明和小红是同事，他们都在腾讯工作",
    "小明是后端工程师，小红是产品经理",
    "他们正在合作一个微信小程序项目",
]

for msg in messages:
    response = conversation.predict(input=msg)
    rprint(f"输入: {msg}")
    rprint(f"AI: {response}\n")

# 测试实体记忆
response = conversation.predict(input="小明是做什么工作的？")
rprint(f"测试: 小明是做什么工作的？")
rprint(f"AI: {response}\n")

# 查看实体记忆
rprint("[cyan]实体记忆:[/cyan]")
rprint(entity_memory.load_memory_variables({}))

## 6. 手动管理记忆

直接操作记忆的保存和加载

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# 手动管理记忆
memory = ConversationBufferMemory(return_messages=True)

rprint("[bold]手动管理记忆[/bold]\n")

# 手动添加对话历史
memory.save_context(
    {"input": "你好，我叫小明"},
    {"output": "你好小明！很高兴认识你。"}
)

memory.save_context(
    {"input": "我在学习LangChain"},
    {"output": "LangChain是一个很好的AI框架，有什么问题可以问我。"}
)

# 查看记忆
rprint("[cyan]记忆内容:[/cyan]")
variables = memory.load_memory_variables({})
for msg in variables["history"]:
    role = "用户" if isinstance(msg, HumanMessage) else "AI"
    rprint(f"  {role}: {msg.content}")

# 清空记忆
memory.clear()
rprint("\n[yellow]记忆已清空[/yellow]")
rprint(f"清空后: {memory.load_memory_variables({})}")

## 7. 在链中使用记忆

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# 创建带记忆的链
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的AI助手。根据对话历史回答问题。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

memory = ConversationBufferMemory(return_messages=True, memory_key="history")

chain = prompt | llm | StrOutputParser()

rprint("[bold]带记忆的链[/bold]\n")

# 模拟对话
def chat(user_input):
    # 加载记忆
    memory_vars = memory.load_memory_variables({})
    # 调用链
    response = chain.invoke({"input": user_input, **memory_vars})
    # 保存到记忆
    memory.save_context({"input": user_input}, {"output": response})
    return response

# 测试对话
rprint(f"用户: 我叫小明")
rprint(f"AI: {chat('我叫小明')}\n")

rprint(f"用户: 我喜欢Python")
rprint(f"AI: {chat('我喜欢Python')}\n")

rprint(f"用户: 你还记得我叫什么吗？")
rprint(f"AI: {chat('你还记得我叫什么吗？')}\n")

rprint(f"用户: 我喜欢什么编程语言？")
rprint(f"AI: {chat('我喜欢什么编程语言？')}")

## 8. 持久化记忆（文件存储）

In [ ]:
import json
from pathlib import Path

class FileMemory:
    """文件持久化记忆"""
    
    def __init__(self, filepath: str):
        self.filepath = Path(filepath)
        self.history = []
        self._load()
    
    def _load(self):
        if self.filepath.exists():
            with open(self.filepath, 'r', encoding='utf-8') as f:
                self.history = json.load(f)
    
    def _save(self):
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)
    
    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content, "timestamp": str(datetime.now())})
        self._save()
    
    def get_history(self) -> list:
        return self.history
    
    def clear(self):
        self.history = []
        self._save()

from datetime import datetime

# 使用文件记忆
file_memory = FileMemory("chat_history.json")

rprint("[bold]文件持久化记忆[/bold]\n")

# 添加对话
file_memory.add("user", "你好，我叫小明")
file_memory.add("assistant", "你好小明！很高兴认识你。")
file_memory.add("user", "今天天气怎么样？")
file_memory.add("assistant", "抱歉，我无法获取实时天气信息。")

# 查看历史
rprint("[cyan]保存的历史:[/cyan]")
for item in file_memory.get_history():
    rprint(f"  [{item['role']}] {item['content']}")

rprint(f"\n[dim]历史已保存到: {file_memory.filepath}[/dim]")

## 9. Redis 持久化记忆

In [ ]:
# Redis 记忆示例（需要安装 redis: pip install redis）
# from langchain.memory import RedisChatMessageHistory

# # 连接 Redis
# redis_history = RedisChatMessageHistory(
#     session_id="user_123",
#     url="redis://localhost:6379"
# )

# # 添加消息
# redis_history.add_user_message("你好，我叫小明")
# redis_history.add_ai_message("你好小明！")

# # 获取消息
# messages = redis_history.messages
# print(messages)

rprint("[bold]Redis 记忆示例[/bold]")
rprint("[dim]需要 Redis 服务和 pip install redis[/dim]")
rprint("[dim]取消注释上方代码运行[/dim]")

## 10. 记忆类型对比

In [ ]:
rprint("""[bold]记忆类型对比[/bold]

| 记忆类型 | 存储方式 | Token消耗 | 适用场景 |
|---------|---------|----------|----------|
| BufferMemory | 全部消息 | 高 | 短对话 |
| BufferWindowMemory | 最近K条 | 中 | 中等对话 |
| SummaryMemory | 摘要 | 低 | 长对话 |
| SummaryBufferMemory | 摘要+最近 | 中低 | 平衡场景 |
| EntityMemory | 实体提取 | 中 | 信息密集对话 |
| FileMemory | 文件存储 | - | 持久化需求 |
| RedisMemory | Redis | - | 分布式系统 |

[bold cyan]选择建议:[/bold cyan]
- 短对话 -> BufferMemory
- 长对话 -> SummaryMemory 或 SummaryBufferMemory
- 需要记住关键信息 -> EntityMemory
- 需要持久化 -> FileMemory 或 RedisMemory
""")